## Step 1: S3 Feature Storage Setup
- Design storage structure for raw and computed features
- Implement versioning with timestamp-based naming
- Create feature metadata organization

In [1]:
import boto3
import json
import pandas as pd
import numpy as np
from datetime import datetime, timezone
import os
from typing import Dict, List, Any
import uuid

# Initialize AWS clients
s3 = boto3.client('s3')
dynamodb = boto3.resource('dynamodb')

# Configuration
FEATURE_BUCKET = 'elbee-oreumi'  # Use existing bucket
FEATURE_PREFIX = 'features/'
RAW_DATA_PREFIX = 'features/raw/'
COMPUTED_PREFIX = 'features/computed/'
METADATA_PREFIX = 'features/metadata/'

# Feature storage structure
print("Feature Storage Structure:")
print(f"Raw Features: s3://{FEATURE_BUCKET}/{RAW_DATA_PREFIX}")
print(f"Computed Features: s3://{FEATURE_BUCKET}/{COMPUTED_PREFIX}")
print(f"Metadata: s3://{FEATURE_BUCKET}/{METADATA_PREFIX}")

# Create sample raw data for feature engineering
sample_data = {
    'customers': [
        {'customer_id': 1, 'age': 25, 'income': 50000, 'city': 'New York', 'signup_date': '2023-01-15'},
        {'customer_id': 2, 'age': 35, 'income': 75000, 'city': 'Los Angeles', 'signup_date': '2023-02-20'},
        {'customer_id': 3, 'age': 45, 'income': 100000, 'city': 'Chicago', 'signup_date': '2023-03-10'},
        {'customer_id': 4, 'age': 28, 'income': 60000, 'city': 'New York', 'signup_date': '2023-04-05'},
        {'customer_id': 5, 'age': 52, 'income': 90000, 'city': 'San Francisco', 'signup_date': '2023-05-12'}
    ],
    'transactions': [
        {'transaction_id': 1, 'customer_id': 1, 'amount': 120.50, 'category': 'electronics', 'date': '2023-06-01'},
        {'transaction_id': 2, 'customer_id': 1, 'amount': 45.20, 'category': 'food', 'date': '2023-06-02'},
        {'transaction_id': 3, 'customer_id': 2, 'amount': 200.00, 'category': 'clothing', 'date': '2023-06-01'},
        {'transaction_id': 4, 'customer_id': 3, 'amount': 89.99, 'category': 'electronics', 'date': '2023-06-03'},
        {'transaction_id': 5, 'customer_id': 2, 'amount': 150.75, 'category': 'food', 'date': '2023-06-04'}
    ]
}

# Upload raw data to S3
for dataset_name, data in sample_data.items():
    key = f"{RAW_DATA_PREFIX}{dataset_name}.json"
    s3.put_object(
        Bucket=FEATURE_BUCKET,
        Key=key,
        Body=json.dumps(data, indent=2)
    )
    print(f"Uploaded raw data: s3://{FEATURE_BUCKET}/{key}")

Feature Storage Structure:
Raw Features: s3://elbee-oreumi/features/raw/
Computed Features: s3://elbee-oreumi/features/computed/
Metadata: s3://elbee-oreumi/features/metadata/
Uploaded raw data: s3://elbee-oreumi/features/raw/customers.json
Uploaded raw data: s3://elbee-oreumi/features/raw/transactions.json


## Step 2: DynamoDB Feature Catalog Setup
- Create table for feature metadata tracking
- Define schema for feature definitions and lineage
- Set up feature discovery capabilities

In [2]:
# Create DynamoDB table for feature catalog (if not exists)
FEATURE_CATALOG_TABLE = 'feature-catalog'

try:
    # Try to create the table
    table = dynamodb.create_table(
        TableName=FEATURE_CATALOG_TABLE,
        KeySchema=[
            {
                'AttributeName': 'feature_name',
                'KeyType': 'HASH'
            }
        ],
        AttributeDefinitions=[
            {
                'AttributeName': 'feature_name',
                'AttributeType': 'S'
            }
        ],
        BillingMode='PAY_PER_REQUEST'  # Free tier friendly
    )
    print(f"Created DynamoDB table: {FEATURE_CATALOG_TABLE}")
    
    # Wait for table to be created
    table.wait_until_exists()
    
except dynamodb.meta.client.exceptions.ResourceInUseException:
    # Table already exists
    table = dynamodb.Table(FEATURE_CATALOG_TABLE)
    print(f"Using existing DynamoDB table: {FEATURE_CATALOG_TABLE}")

# Feature catalog helper functions
class FeatureCatalog:
    def __init__(self, table_name):
        self.table = dynamodb.Table(table_name)
    
    def register_feature(self, feature_name: str, feature_definition: Dict[str, Any]):
        """Register a new feature in the catalog"""
        item = {
            'feature_name': feature_name,
            'created_at': datetime.now(timezone.utc).isoformat(),
            'updated_at': datetime.now(timezone.utc).isoformat(),
            **feature_definition
        }
        
        self.table.put_item(Item=item)
        print(f"Registered feature: {feature_name}")
        return item
    
    def get_feature(self, feature_name: str):
        """Get feature definition from catalog"""
        response = self.table.get_item(Key={'feature_name': feature_name})
        return response.get('Item')
    
    def list_features(self, category: str = None):
        """List all features, optionally filtered by category"""
        response = self.table.scan()
        features = response.get('Items', [])
        
        if category:
            features = [f for f in features if f.get('category') == category]
            
        return features
    
    def update_feature_stats(self, feature_name: str, stats: Dict[str, Any]):
        """Update feature usage statistics"""
        self.table.update_item(
            Key={'feature_name': feature_name},
            UpdateExpression='SET updated_at = :updated_at, stats = :stats',
            ExpressionAttributeValues={
                ':updated_at': datetime.now(timezone.utc).isoformat(),
                ':stats': stats
            }
        )

# Initialize catalog
catalog = FeatureCatalog(FEATURE_CATALOG_TABLE)
print("Feature catalog initialized successfully")

Created DynamoDB table: feature-catalog
Feature catalog initialized successfully


## Step 3: Lambda Feature Processing Functions
- Create feature transformation functions
- Implement scalable feature engineering logic
- Add error handling and validation

In [ ]:
# Feature Engineering Functions (Lambda-compatible)
import math
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

class FeatureEngineer:
    """Feature engineering functions for Lambda processing"""
    
    @staticmethod
    def age_categories(age: float) -> str:
        """Convert age to categorical features"""
        if age < 25:
            return 'young'
        elif age < 35:
            return 'adult'
        elif age < 50:
            return 'middle_aged'
        else:
            return 'senior'
    
    @staticmethod
    def income_tier(income: float) -> str:
        """Categorize income into tiers"""
        if income < 40000:
            return 'low'
        elif income < 70000:
            return 'medium'
        elif income < 100000:
            return 'high'
        else:
            return 'premium'
    
    @staticmethod
    def customer_tenure_days(signup_date: str) -> int:
        """Calculate customer tenure in days"""
        signup = datetime.fromisoformat(signup_date)
        today = datetime.now()
        return (today - signup).days
    
    @staticmethod
    def transaction_frequency_features(transactions: List[Dict]) -> Dict[str, float]:
        """Calculate transaction frequency features"""
        if not transactions:
            return {'txn_count': 0, 'avg_amount': 0, 'total_amount': 0}
        
        amounts = [t['amount'] for t in transactions]
        return {
            'txn_count': len(transactions),
            'avg_amount': sum(amounts) / len(amounts),
            'total_amount': sum(amounts),
            'max_amount': max(amounts),
            'min_amount': min(amounts)
        }
    
    @staticmethod
    def categorical_spending_features(transactions: List[Dict]) -> Dict[str, float]:
        """Calculate spending by category"""
        category_spending = {}
        for txn in transactions:
            category = txn['category']
            category_spending[category] = category_spending.get(category, 0) + txn['amount']
        
        # Convert to feature format
        features = {}
        for category, amount in category_spending.items():
            features[f'spending_{category}'] = amount
        
        return features

# Test feature engineering functions
print("Testing Feature Engineering Functions:")
fe = FeatureEngineer()

# Test with sample data
print(f"Age 30 -> {fe.age_categories(30)}")
print(f"Income 65000 -> {fe.income_tier(65000)}")
print(f"Signup '2023-01-15' -> {fe.customer_tenure_days('2023-01-15')} days")

sample_transactions = [
    {'amount': 100, 'category': 'electronics'},
    {'amount': 50, 'category': 'food'},
    {'amount': 200, 'category': 'electronics'}
]
freq_features = fe.transaction_frequency_features(sample_transactions)
cat_features = fe.categorical_spending_features(sample_transactions)

print(f"Frequency features: {freq_features}")
print(f"Category features: {cat_features}")

## Step 3.1: Local Dry Run Testing
- Test feature engineering logic locally without AWS calls
- Debug data transformations and validations
- Mock AWS services for offline development

In [6]:
# Dry Run Testing - Local Feature Engineering Pipeline
import io
from unittest.mock import Mock, patch, MagicMock
import tempfile
import os

class DryRunTester:
    """Test feature engineering pipeline locally without AWS calls"""
    
    def __init__(self):
        self.mock_s3_data = {}
        self.mock_dynamodb_data = {}
        self.debug_logs = []
    
    def log_debug(self, message):
        """Add debug message to logs"""
        timestamp = datetime.now().strftime("%H:%M:%S")
        self.debug_logs.append(f"[{timestamp}] {message}")
        print(f"🔍 DEBUG: {message}")
    
    def setup_mock_data(self):
        """Setup mock data for testing"""
        # Mock raw data (same as what we upload to S3)
        self.mock_s3_data = {
            'features/raw/customers.json': {
                'Body': io.BytesIO(json.dumps([
                    {'customer_id': 1, 'age': 25, 'income': 50000, 'city': 'New York', 'signup_date': '2023-01-15'},
                    {'customer_id': 2, 'age': 35, 'income': 75000, 'city': 'Los Angeles', 'signup_date': '2023-02-20'},
                    {'customer_id': 3, 'age': 45, 'income': 100000, 'city': 'Chicago', 'signup_date': '2023-03-10'}
                ]).encode())
            },
            'features/raw/transactions.json': {
                'Body': io.BytesIO(json.dumps([
                    {'transaction_id': 1, 'customer_id': 1, 'amount': 120.50, 'category': 'electronics', 'date': '2023-06-01'},
                    {'transaction_id': 2, 'customer_id': 1, 'amount': 45.20, 'category': 'food', 'date': '2023-06-02'},
                    {'transaction_id': 3, 'customer_id': 2, 'amount': 200.00, 'category': 'clothing', 'date': '2023-06-01'},
                    {'transaction_id': 4, 'customer_id': 3, 'amount': 89.99, 'category': 'electronics', 'date': '2023-06-03'}
                ]).encode())
            }
        }
        
        self.log_debug("Mock data setup completed")
    
    def mock_s3_get_object(self, **kwargs):
        """Mock S3 get_object calls"""
        bucket = kwargs.get('Bucket')
        key = kwargs.get('Key')
        full_key = key
        
        self.log_debug(f"S3 GET: s3://{bucket}/{key}")
        
        if full_key in self.mock_s3_data:
            # Reset BytesIO position for reading
            self.mock_s3_data[full_key]['Body'].seek(0)
            return self.mock_s3_data[full_key]
        else:
            raise Exception(f"Mock S3 object not found: {full_key}")
    
    def mock_s3_put_object(self, **kwargs):
        """Mock S3 put_object calls"""
        bucket = kwargs.get('Bucket')
        key = kwargs.get('Key')
        body = kwargs.get('Body')
        
        self.log_debug(f"S3 PUT: s3://{bucket}/{key} ({len(str(body))} bytes)")
        
        # Store the uploaded data for inspection
        if isinstance(body, str):
            self.mock_s3_data[key] = {'Body': io.BytesIO(body.encode())}
        else:
            self.mock_s3_data[key] = {'Body': io.BytesIO(str(body).encode())}
        
        return {'ETag': 'mock-etag'}
    
    def mock_dynamodb_put_item(self, **kwargs):
        """Mock DynamoDB put_item calls"""
        item = kwargs.get('Item')
        feature_name = item.get('feature_name')
        
        self.log_debug(f"DynamoDB PUT: {feature_name}")
        self.mock_dynamodb_data[feature_name] = item
        
        return {'ResponseMetadata': {'HTTPStatusCode': 200}}
    
    def run_dry_run_pipeline(self):
        """Run the feature processing pipeline with mocked AWS services"""
        self.log_debug("Starting dry run feature processing pipeline")
        
        # Setup mock data
        self.setup_mock_data()
        
        # Mock AWS clients
        with patch('boto3.client') as mock_s3_client, \
             patch('boto3.resource') as mock_dynamodb_resource:
            
            # Setup S3 mock
            mock_s3 = Mock()
            mock_s3.get_object.side_effect = self.mock_s3_get_object
            mock_s3.put_object.side_effect = self.mock_s3_put_object
            mock_s3_client.return_value = mock_s3
            
            # Setup DynamoDB mock
            mock_table = Mock()
            mock_table.put_item.side_effect = self.mock_dynamodb_put_item
            mock_dynamodb = Mock()
            mock_dynamodb.Table.return_value = mock_table
            mock_dynamodb_resource.return_value = mock_dynamodb
            
            try:
                # Test individual feature engineering functions
                self.log_debug("Testing individual feature functions")
                fe = FeatureEngineer()
                
                # Test age categories
                age_test = fe.age_categories(30)
                self.log_debug(f"Age 30 categorized as: {age_test}")
                assert age_test == 'adult', f"Expected 'adult', got '{age_test}'"
                
                # Test income tier
                income_test = fe.income_tier(65000)
                self.log_debug(f"Income 65000 categorized as: {income_test}")
                assert income_test == 'medium', f"Expected 'medium', got '{income_test}'"
                
                # Test transaction features
                sample_txns = [
                    {'amount': 100, 'category': 'electronics'},
                    {'amount': 50, 'category': 'food'}
                ]
                freq_features = fe.transaction_frequency_features(sample_txns)
                self.log_debug(f"Transaction frequency features: {freq_features}")
                
                # Load and process mock data
                self.log_debug("Processing customer features with mock data")
                
                # Mock the process_customer_features function logic
                customers_data = json.loads(self.mock_s3_data['features/raw/customers.json']['Body'].read())
                transactions_data = json.loads(self.mock_s3_data['features/raw/transactions.json']['Body'].read())
                
                # Reset BytesIO for next read
                self.mock_s3_data['features/raw/customers.json']['Body'].seek(0)
                self.mock_s3_data['features/raw/transactions.json']['Body'].seek(0)
                
                customers_df = pd.DataFrame(customers_data)
                transactions_df = pd.DataFrame(transactions_data)
                
                self.log_debug(f"Loaded {len(customers_df)} customers, {len(transactions_df)} transactions")
                
                # Process features for each customer
                customer_features = []
                
                for _, customer in customers_df.iterrows():
                    customer_id = customer['customer_id']
                    
                    # Basic features
                    features = {
                        'customer_id': customer_id,
                        'age': customer['age'],
                        'income': customer['income'],
                        'city': customer['city']
                    }
                    
                    # Engineered features
                    features['age_category'] = fe.age_categories(customer['age'])
                    features['income_tier'] = fe.income_tier(customer['income'])
                    features['tenure_days'] = fe.customer_tenure_days(customer['signup_date'])
                    
                    # Transaction-based features
                    customer_transactions = transactions_df[
                        transactions_df['customer_id'] == customer_id
                    ].to_dict('records')
                    
                    freq_features = fe.transaction_frequency_features(customer_transactions)
                    cat_features = fe.categorical_spending_features(customer_transactions)
                    
                    features.update(freq_features)
                    features.update(cat_features)
                    
                    customer_features.append(features)
                    self.log_debug(f"Processed customer {customer_id}: {len(features)} features")
                
                # Convert to DataFrame
                features_df = pd.DataFrame(customer_features)
                self.log_debug(f"Created feature DataFrame: {features_df.shape}")
                
                # Test data quality
                self.log_debug("Running data quality checks")
                quality_report = {
                    'record_count': len(features_df),
                    'feature_count': len(features_df.columns),
                    'missing_values': features_df.isnull().sum().sum(),
                    'duplicate_records': features_df.duplicated().sum()
                }
                
                self.log_debug(f"Quality report: {quality_report}")
                
                # Simulate S3 upload
                timestamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
                csv_data = features_df.to_csv(index=False)
                json_data = features_df.to_json(orient='records', indent=2)
                
                # Mock upload calls
                mock_s3.put_object(
                    Bucket='test-bucket',
                    Key=f'features/computed/customer_features_{timestamp}.csv',
                    Body=csv_data
                )
                
                mock_s3.put_object(
                    Bucket='test-bucket',
                    Key=f'features/computed/customer_features_{timestamp}.json',
                    Body=json_data
                )
                
                # Mock DynamoDB catalog update
                feature_definition = {
                    'feature_name': 'customer_features',
                    'category': 'customer',
                    'description': 'Test customer features',
                    'feature_count': len(features_df.columns),
                    'features': list(features_df.columns)
                }
                
                mock_table.put_item(Item=feature_definition)
                
                self.log_debug("✅ Dry run completed successfully")
                
                return {
                    'status': 'success',
                    'features_generated': len(features_df.columns),
                    'customers_processed': len(features_df),
                    'quality_report': quality_report,
                    'sample_features': features_df.head(2).to_dict('records')
                }
                
            except Exception as e:
                self.log_debug(f"❌ Dry run failed: {e}")
                return {
                    'status': 'error',
                    'error': str(e),
                    'debug_logs': self.debug_logs
                }
    
    def print_debug_summary(self):
        """Print summary of all debug logs"""
        print("\n" + "="*50)
        print("🔍 DRY RUN DEBUG SUMMARY")
        print("="*50)
        for log in self.debug_logs:
            print(log)
        print("="*50)

# Run the dry run test
print("🧪 Starting Feature Engineering Dry Run Test")
print("This will test the pipeline locally without AWS calls\n")

dry_run = DryRunTester()
result = dry_run.run_dry_run_pipeline()

print("\n📊 Dry Run Results:")
print(json.dumps(result, indent=2))

if result['status'] == 'success':
    print("\n✅ All tests passed! The pipeline is ready for AWS deployment.")
    print(f"\n📈 Pipeline Summary:")
    print(f"   - Features Generated: {result['features_generated']}")
    print(f"   - Customers Processed: {result['customers_processed']}")
    print(f"   - Missing Values: {result['quality_report']['missing_values']}")
    print(f"   - Duplicate Records: {result['quality_report']['duplicate_records']}")
    
    print("\n🎯 Sample Features Generated:")
    for i, sample in enumerate(result['sample_features'][:1], 1):
        print(f"   Customer {i}:")
        for feature, value in list(sample.items())[:8]:  # Show first 8 features
            print(f"     {feature}: {value}")
        if len(sample) > 8:
            print(f"     ... and {len(sample) - 8} more features")
else:
    print("\n❌ Tests failed! Check the debug logs for details.")
    if 'error' in result:
        print(f"Error: {result['error']}")

# Show debug summary
print("\n🔧 Want to see detailed debug logs? Uncomment the line below:")
print("# dry_run.print_debug_summary()")

🧪 Starting Feature Engineering Dry Run Test
This will test the pipeline locally without AWS calls

🔍 DEBUG: Starting dry run feature processing pipeline
🔍 DEBUG: Mock data setup completed
🔍 DEBUG: Testing individual feature functions
🔍 DEBUG: ❌ Dry run failed: name 'FeatureEngineer' is not defined

📊 Dry Run Results:
{
  "status": "error",
  "error": "name 'FeatureEngineer' is not defined",
  "debug_logs": [
    "[03:25:20] Starting dry run feature processing pipeline",
    "[03:25:20] Mock data setup completed",
    "[03:25:20] Testing individual feature functions",
    "[03:25:20] \u274c Dry run failed: name 'FeatureEngineer' is not defined"
  ]
}

❌ Tests failed! Check the debug logs for details.
Error: name 'FeatureEngineer' is not defined

🔧 Want to see detailed debug logs? Uncomment the line below:
# dry_run.print_debug_summary()


## Step 4: Feature Processing Pipeline
- Process raw data and generate features
- Store computed features with versioning
- Update feature catalog with metadata

In [ ]:
def process_customer_features():
    """Main feature processing function (Lambda handler-compatible)"""
    
    # Load raw data from S3
    customers_obj = s3.get_object(Bucket=FEATURE_BUCKET, Key=f'{RAW_DATA_PREFIX}customers.json')
    customers_data = json.loads(customers_obj['Body'].read())
    
    transactions_obj = s3.get_object(Bucket=FEATURE_BUCKET, Key=f'{RAW_DATA_PREFIX}transactions.json')
    transactions_data = json.loads(transactions_obj['Body'].read())
    
    # Convert to DataFrames for easier processing
    customers_df = pd.DataFrame(customers_data)
    transactions_df = pd.DataFrame(transactions_data)
    
    # Initialize feature engineer
    fe = FeatureEngineer()
    
    # Generate customer features
    customer_features = []
    
    for _, customer in customers_df.iterrows():
        customer_id = customer['customer_id']
        
        # Basic demographic features
        features = {
            'customer_id': customer_id,
            'age': customer['age'],
            'income': customer['income'],
            'city': customer['city']
        }
        
        # Engineered demographic features
        features['age_category'] = fe.age_categories(customer['age'])
        features['income_tier'] = fe.income_tier(customer['income'])
        features['tenure_days'] = fe.customer_tenure_days(customer['signup_date'])
        
        # Get customer transactions
        customer_transactions = transactions_df[
            transactions_df['customer_id'] == customer_id
        ].to_dict('records')
        
        # Transaction-based features
        freq_features = fe.transaction_frequency_features(customer_transactions)
        cat_features = fe.categorical_spending_features(customer_transactions)
        
        # Combine all features
        features.update(freq_features)
        features.update(cat_features)
        
        customer_features.append(features)
    
    # Convert to DataFrame and save
    features_df = pd.DataFrame(customer_features)
    
    # Generate timestamp for versioning
    timestamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
    
    # Save to S3 as CSV and JSON
    csv_key = f'{COMPUTED_PREFIX}customer_features_{timestamp}.csv'
    json_key = f'{COMPUTED_PREFIX}customer_features_{timestamp}.json'
    
    # Upload CSV
    csv_buffer = features_df.to_csv(index=False)
    s3.put_object(Bucket=FEATURE_BUCKET, Key=csv_key, Body=csv_buffer)
    
    # Upload JSON
    json_buffer = features_df.to_json(orient='records', indent=2)
    s3.put_object(Bucket=FEATURE_BUCKET, Key=json_key, Body=json_buffer)
    
    print(f"Features saved to:")
    print(f"  CSV: s3://{FEATURE_BUCKET}/{csv_key}")
    print(f"  JSON: s3://{FEATURE_BUCKET}/{json_key}")
    
    # Update feature catalog
    feature_definitions = {
        'customer_features': {
            'category': 'customer',
            'description': 'Comprehensive customer features including demographics and transaction patterns',
            'source_data': ['customers.json', 'transactions.json'],
            'feature_count': len(features_df.columns),
            'features': list(features_df.columns),
            's3_location_csv': f's3://{FEATURE_BUCKET}/{csv_key}',
            's3_location_json': f's3://{FEATURE_BUCKET}/{json_key}',
            'schema': {col: str(features_df[col].dtype) for col in features_df.columns},
            'sample_data': features_df.head(2).to_dict('records'),
            'stats': {
                'record_count': len(features_df),
                'null_counts': features_df.isnull().sum().to_dict()
            }
        }
    }
    
    # Register in catalog
    for feature_name, definition in feature_definitions.items():
        catalog.register_feature(feature_name, definition)
    
    return {
        'statusCode': 200,
        'body': f'Processed {len(features_df)} customer feature records',
        'feature_locations': {
            'csv': f's3://{FEATURE_BUCKET}/{csv_key}',
            'json': f's3://{FEATURE_BUCKET}/{json_key}'
        }
    }

# Run the feature processing pipeline
result = process_customer_features()
print("\nFeature Processing Result:")
print(json.dumps(result, indent=2))

## Step 5: Feature Discovery Interface
- Query the DynamoDB feature catalog
- Download and explore feature datasets
- Provide feature selection capabilities

In [ ]:
class FeatureDiscovery:
    """Interface for discovering and using features"""
    
    def __init__(self, catalog: FeatureCatalog, bucket: str):
        self.catalog = catalog
        self.bucket = bucket
        self.s3 = boto3.client('s3')
    
    def discover_features(self, category: str = None) -> pd.DataFrame:
        """Discover available features"""
        features = self.catalog.list_features(category)
        
        if not features:
            print("No features found")
            return pd.DataFrame()
        
        # Convert to readable format
        discovery_data = []
        for feature in features:
            discovery_data.append({
                'Feature Name': feature['feature_name'],
                'Category': feature.get('category', 'unknown'),
                'Description': feature.get('description', ''),
                'Feature Count': feature.get('feature_count', 0),
                'Record Count': feature.get('stats', {}).get('record_count', 0),
                'Created': feature.get('created_at', ''),
                'Updated': feature.get('updated_at', '')
            })
        
        return pd.DataFrame(discovery_data)
    
    def get_feature_details(self, feature_name: str) -> Dict[str, Any]:
        """Get detailed information about a specific feature"""
        return self.catalog.get_feature(feature_name)
    
    def load_feature_data(self, feature_name: str, format: str = 'csv') -> pd.DataFrame:
        """Load feature data from S3"""
        feature_info = self.catalog.get_feature(feature_name)
        if not feature_info:
            raise ValueError(f"Feature {feature_name} not found in catalog")
        
        # Get S3 location
        s3_key = None
        if format == 'csv' and 's3_location_csv' in feature_info:
            s3_location = feature_info['s3_location_csv']
            s3_key = s3_location.replace(f's3://{self.bucket}/', '')
        elif format == 'json' and 's3_location_json' in feature_info:
            s3_location = feature_info['s3_location_json']
            s3_key = s3_location.replace(f's3://{self.bucket}/', '')
        
        if not s3_key:
            raise ValueError(f"No {format} location found for feature {feature_name}")
        
        # Download and load data
        obj = self.s3.get_object(Bucket=self.bucket, Key=s3_key)
        
        if format == 'csv':
            return pd.read_csv(obj['Body'])
        elif format == 'json':
            data = json.loads(obj['Body'].read())
            return pd.DataFrame(data)
    
    def search_features(self, search_term: str) -> pd.DataFrame:
        """Search features by name or description"""
        all_features = self.catalog.list_features()
        
        matching_features = []
        search_term_lower = search_term.lower()
        
        for feature in all_features:
            feature_name = feature['feature_name'].lower()
            description = feature.get('description', '').lower()
            
            if search_term_lower in feature_name or search_term_lower in description:
                matching_features.append(feature)
        
        if not matching_features:
            print(f"No features found matching '{search_term}'")
            return pd.DataFrame()
        
        # Convert to readable format
        discovery_data = []
        for feature in matching_features:
            discovery_data.append({
                'Feature Name': feature['feature_name'],
                'Category': feature.get('category', 'unknown'),
                'Description': feature.get('description', ''),
                'Features': ', '.join(feature.get('features', [])[:5])  # Show first 5 features
            })
        
        return pd.DataFrame(discovery_data)

# Initialize discovery interface
discovery = FeatureDiscovery(catalog, FEATURE_BUCKET)

print("=== Feature Discovery Interface ===")
print("\n1. Discover All Features:")
all_features = discovery.discover_features()
print(all_features.to_string(index=False) if not all_features.empty else "No features found")

print("\n2. Get Feature Details:")
if not all_features.empty:
    feature_name = all_features.iloc[0]['Feature Name']
    details = discovery.get_feature_details(feature_name)
    print(f"Details for '{feature_name}':")
    print(f"  Category: {details.get('category')}")
    print(f"  Description: {details.get('description')}")
    print(f"  Features: {', '.join(details.get('features', [])[:5])}{'...' if len(details.get('features', [])) > 5 else ''}")
    print(f"  Records: {details.get('stats', {}).get('record_count')}")

print("\n3. Search Features:")
search_results = discovery.search_features('customer')
print(search_results.to_string(index=False) if not search_results.empty else "No matching features found")

## Step 6: Load and Explore Feature Data
- Load computed features from S3
- Perform basic feature analysis
- Demonstrate feature selection for ML

In [ ]:
# Load and explore the computed features
print("=== Feature Data Exploration ===")

try:
    # Load customer features
    customer_features_df = discovery.load_feature_data('customer_features', 'csv')
    
    print(f"\nLoaded {len(customer_features_df)} customer feature records")
    print(f"Feature columns ({len(customer_features_df.columns)}): {list(customer_features_df.columns)}")
    
    print("\nSample feature data:")
    print(customer_features_df.head())
    
    print("\nFeature statistics:")
    print(customer_features_df.describe(include='all'))
    
    print("\nData types:")
    print(customer_features_df.dtypes)
    
    print("\nMissing values:")
    missing_values = customer_features_df.isnull().sum()
    print(missing_values[missing_values > 0] if missing_values.sum() > 0 else "No missing values")
    
    # Feature selection example for ML
    print("\n=== Feature Selection for ML ===")
    
    # Select numerical features for a hypothetical ML model
    numerical_features = customer_features_df.select_dtypes(include=[np.number]).columns.tolist()
    print(f"Numerical features: {numerical_features}")
    
    # Select categorical features
    categorical_features = customer_features_df.select_dtypes(include=['object']).columns.tolist()
    categorical_features = [f for f in categorical_features if f != 'customer_id']  # Exclude ID
    print(f"Categorical features: {categorical_features}")
    
    # Example feature set for customer segmentation
    segmentation_features = ['age', 'income', 'tenure_days', 'txn_count', 'avg_amount', 'total_amount']
    available_segmentation_features = [f for f in segmentation_features if f in customer_features_df.columns]
    
    if available_segmentation_features:
        print(f"\nCustomer segmentation feature set: {available_segmentation_features}")
        segmentation_data = customer_features_df[['customer_id'] + available_segmentation_features]
        print(segmentation_data.head())
        
        # Basic correlation analysis
        correlation_matrix = customer_features_df[available_segmentation_features].corr()
        print("\nFeature correlations:")
        print(correlation_matrix)
    
except Exception as e:
    print(f"Error loading feature data: {e}")
    print("Make sure the feature processing pipeline has been run successfully.")

## Step 7: Batch Feature Pipeline Simulation
- Simulate scheduled feature processing
- Implement error handling and retry logic
- Basic monitoring and quality checks

In [ ]:
import time
import traceback
from typing import Optional

class BatchFeaturePipeline:
    """Batch processing pipeline for features"""
    
    def __init__(self, catalog: FeatureCatalog, bucket: str):
        self.catalog = catalog
        self.bucket = bucket
        self.s3 = boto3.client('s3')
        self.processing_log = []
    
    def validate_raw_data(self, data_key: str) -> bool:
        """Validate raw data before processing"""
        try:
            obj = self.s3.get_object(Bucket=self.bucket, Key=data_key)
            data = json.loads(obj['Body'].read())
            
            # Basic validation
            if not isinstance(data, list) or len(data) == 0:
                return False
                
            # Check required fields based on data type
            if 'customers' in data_key:
                required_fields = ['customer_id', 'age', 'income']
            elif 'transactions' in data_key:
                required_fields = ['transaction_id', 'customer_id', 'amount']
            else:
                return True  # Unknown data type, assume valid
            
            # Validate first record has required fields
            first_record = data[0]
            for field in required_fields:
                if field not in first_record:
                    return False
            
            return True
            
        except Exception as e:
            print(f"Validation error for {data_key}: {e}")
            return False
    
    def run_quality_checks(self, feature_data: pd.DataFrame) -> Dict[str, Any]:
        """Run data quality checks on computed features"""
        quality_report = {
            'record_count': len(feature_data),
            'feature_count': len(feature_data.columns),
            'missing_values': feature_data.isnull().sum().to_dict(),
            'duplicate_records': feature_data.duplicated().sum(),
            'data_types': feature_data.dtypes.to_dict()
        }
        
        # Check for outliers in numerical columns
        numerical_cols = feature_data.select_dtypes(include=[np.number]).columns
        outliers = {}
        for col in numerical_cols:
            if len(feature_data[col].dropna()) > 0:
                Q1 = feature_data[col].quantile(0.25)
                Q3 = feature_data[col].quantile(0.75)
                IQR = Q3 - Q1
                lower_bound = Q1 - 1.5 * IQR
                upper_bound = Q3 + 1.5 * IQR
                outliers[col] = ((feature_data[col] < lower_bound) | (feature_data[col] > upper_bound)).sum()
        
        quality_report['outliers'] = outliers
        
        return quality_report
    
    def process_with_retry(self, max_retries: int = 3) -> Dict[str, Any]:
        """Process features with retry logic"""
        
        for attempt in range(max_retries):
            try:
                print(f"\nProcessing attempt {attempt + 1}/{max_retries}")
                
                # Step 1: Validate raw data
                print("Step 1: Validating raw data...")
                raw_data_keys = [f'{RAW_DATA_PREFIX}customers.json', f'{RAW_DATA_PREFIX}transactions.json']
                
                for key in raw_data_keys:
                    if not self.validate_raw_data(key):
                        raise ValueError(f"Data validation failed for {key}")
                
                print("✓ Raw data validation passed")
                
                # Step 2: Process features
                print("Step 2: Processing features...")
                result = process_customer_features()
                
                # Step 3: Load and validate computed features
                print("Step 3: Validating computed features...")
                feature_data = discovery.load_feature_data('customer_features', 'csv')
                
                # Step 4: Run quality checks
                print("Step 4: Running quality checks...")
                quality_report = self.run_quality_checks(feature_data)
                
                # Step 5: Log successful processing
                processing_record = {
                    'timestamp': datetime.now(timezone.utc).isoformat(),
                    'status': 'success',
                    'attempt': attempt + 1,
                    'result': result,
                    'quality_report': quality_report
                }
                
                self.processing_log.append(processing_record)
                
                print("✓ Feature processing completed successfully")
                return processing_record
                
            except Exception as e:
                error_record = {
                    'timestamp': datetime.now(timezone.utc).isoformat(),
                    'status': 'error',
                    'attempt': attempt + 1,
                    'error': str(e),
                    'traceback': traceback.format_exc()
                }
                
                self.processing_log.append(error_record)
                print(f"✗ Error in attempt {attempt + 1}: {e}")
                
                if attempt < max_retries - 1:
                    print(f"Retrying in 2 seconds...")
                    time.sleep(2)
        
        print(f"✗ All {max_retries} attempts failed")
        return {'status': 'failed', 'attempts': max_retries}
    
    def get_processing_summary(self) -> Dict[str, Any]:
        """Get summary of processing runs"""
        if not self.processing_log:
            return {'message': 'No processing runs recorded'}
        
        successful_runs = [log for log in self.processing_log if log['status'] == 'success']
        failed_runs = [log for log in self.processing_log if log['status'] == 'error']
        
        summary = {
            'total_runs': len(self.processing_log),
            'successful_runs': len(successful_runs),
            'failed_runs': len(failed_runs),
            'success_rate': len(successful_runs) / len(self.processing_log) * 100,
            'last_run': self.processing_log[-1]['timestamp'] if self.processing_log else None,
            'last_status': self.processing_log[-1]['status'] if self.processing_log else None
        }
        
        if successful_runs:
            last_success = successful_runs[-1]
            summary['last_successful_run'] = last_success['timestamp']
            if 'quality_report' in last_success:
                summary['last_quality_report'] = last_success['quality_report']
        
        return summary

# Run the batch pipeline
print("=== Batch Feature Pipeline ===")
pipeline = BatchFeaturePipeline(catalog, FEATURE_BUCKET)

# Simulate scheduled processing
result = pipeline.process_with_retry(max_retries=2)

print("\n=== Processing Summary ===")
summary = pipeline.get_processing_summary()
for key, value in summary.items():
    if isinstance(value, dict):
        print(f"{key}:")
        for sub_key, sub_value in value.items():
            print(f"  {sub_key}: {sub_value}")
    else:
        print(f"{key}: {value}")

## Step 8: Lambda Deployment Template
- Provide deployment code for AWS Lambda
- Include IAM permissions and environment setup
- CloudWatch Events scheduling example

In [ ]:
# Lambda deployment template
lambda_handler_code = '''
import json
import boto3
import pandas as pd
import numpy as np
from datetime import datetime, timezone
from typing import Dict, List, Any

# Initialize AWS clients
s3 = boto3.client('s3')
dynamodb = boto3.resource('dynamodb')

# Configuration (set via environment variables)
FEATURE_BUCKET = os.environ.get('FEATURE_BUCKET')
FEATURE_CATALOG_TABLE = os.environ.get('FEATURE_CATALOG_TABLE')
RAW_DATA_PREFIX = 'features/raw/'
COMPUTED_PREFIX = 'features/computed/'

# [Include all FeatureEngineer class code here]

def lambda_handler(event, context):
    """Lambda handler for feature processing"""
    try:
        # Process features
        result = process_customer_features()
        
        return {
            'statusCode': 200,
            'body': json.dumps(result)
        }
        
    except Exception as e:
        print(f"Error processing features: {e}")
        return {
            'statusCode': 500,
            'body': json.dumps({
                'error': str(e)
            })
        }

# [Include process_customer_features function here]
'''

# IAM policy for Lambda execution
iam_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "s3:GetObject",
                "s3:PutObject",
                "s3:ListBucket"
            ],
            "Resource": [
                f"arn:aws:s3:::{FEATURE_BUCKET}",
                f"arn:aws:s3:::{FEATURE_BUCKET}/*"
            ]
        },
        {
            "Effect": "Allow",
            "Action": [
                "dynamodb:PutItem",
                "dynamodb:GetItem",
                "dynamodb:UpdateItem",
                "dynamodb:Scan"
            ],
            "Resource": f"arn:aws:dynamodb:*:*:table/{FEATURE_CATALOG_TABLE}"
        },
        {
            "Effect": "Allow",
            "Action": [
                "logs:CreateLogGroup",
                "logs:CreateLogStream",
                "logs:PutLogEvents"
            ],
            "Resource": "arn:aws:logs:*:*:*"
        }
    ]
}

# CloudWatch Events rule for scheduling (cron)
cloudwatch_event_rule = {
    "Name": "feature-processing-schedule",
    "Description": "Schedule feature processing every day at 2 AM",
    "ScheduleExpression": "cron(0 2 * * ? *)",  # Daily at 2 AM UTC
    "State": "ENABLED"
}

print("=== Lambda Deployment Configuration ===")
print("\n1. Lambda Function Code:")
print("Save the above lambda_handler_code to a .py file and create a deployment package")

print("\n2. Required Environment Variables:")
print(f"FEATURE_BUCKET={FEATURE_BUCKET}")
print(f"FEATURE_CATALOG_TABLE={FEATURE_CATALOG_TABLE}")

print("\n3. IAM Policy (attach to Lambda execution role):")
print(json.dumps(iam_policy, indent=2))

print("\n4. CloudWatch Events Rule (for scheduling):")
print(json.dumps(cloudwatch_event_rule, indent=2))

print("\n5. Deployment Commands:")
print("# Create deployment package")
print("pip install pandas numpy -t ./lambda_package/")
print("cp lambda_function.py lambda_package/")
print("cd lambda_package && zip -r ../feature_processor.zip .")
print("\n# Deploy using AWS CLI")
print("aws lambda create-function --function-name feature-processor \\")
print("  --runtime python3.9 --role arn:aws:iam::ACCOUNT:role/lambda-execution-role \\")
print("  --handler lambda_function.lambda_handler \\")
print("  --zip-file fileb://feature_processor.zip \\")
print("  --timeout 300")

print("\n=== Free Tier Considerations ===")
print("✓ Lambda: 1M free requests/month")
print("✓ DynamoDB: 25GB free storage")
print("✓ S3: 5GB free storage")
print("✓ CloudWatch: Basic monitoring included")
print("\n⚠️  Monitor usage to stay within limits")
print("⚠️  Use small datasets for testing")
print("⚠️  Clean up old feature versions regularly")

## Summary and Cleanup
- Review what was implemented
- Cleanup temporary resources
- Free tier usage optimization tips

In [ ]:
print("=== Mini Project 3 Summary ===")
print("\n✅ Implemented Components:")
print("1. S3 Feature Storage with versioned organization")
print("2. DynamoDB Feature Catalog for metadata tracking")
print("3. Feature Engineering functions (demographics, transactions)")
print("4. Automated feature processing pipeline")
print("5. Feature discovery and exploration interface")
print("6. Batch processing with retry logic and quality checks")
print("7. Lambda deployment template with IAM policies")

print("\n🎯 Learning Objectives Achieved:")
print("✓ Implemented feature engineering workflows")
print("✓ Created reusable feature transformations")
print("✓ Built feature storage and versioning system")
print("✓ Developed feature discovery mechanisms")
print("✓ Set up metadata management with DynamoDB")
print("✓ Demonstrated error handling and quality checks")

print("\n📊 Generated Features:")
try:
    feature_data = discovery.load_feature_data('customer_features', 'csv')
    print(f"Total Features: {len(feature_data.columns)}")
    print(f"Customer Records: {len(feature_data)}")
    print(f"Feature Types: {list(feature_data.select_dtypes(include=[np.number]).columns)[:5]}...")
except:
    print("Feature data not available - run the processing pipeline first")

print("\n💰 Free Tier Usage:")
print("S3 Storage: ~1MB (raw data + computed features)")
print("DynamoDB: ~1KB (feature catalog metadata)")
print("Lambda: 0 requests (local execution only)")
print("Total Cost: $0.00 (within free tier limits)")

print("\n🔧 Next Steps:")
print("1. Deploy Lambda function for automated processing")
print("2. Set up CloudWatch Events for scheduling")
print("3. Add more sophisticated feature engineering")
print("4. Integrate with ML model training pipelines")
print("5. Implement feature serving for real-time inference")

# Optional: Cleanup resources
cleanup_choice = input("\nDo you want to clean up S3 test data? (y/n): ")
if cleanup_choice.lower() == 'y':
    try:
        # List and delete feature files
        response = s3.list_objects_v2(Bucket=FEATURE_BUCKET, Prefix=FEATURE_PREFIX)
        if 'Contents' in response:
            for obj in response['Contents']:
                s3.delete_object(Bucket=FEATURE_BUCKET, Key=obj['Key'])
                print(f"Deleted: s3://{FEATURE_BUCKET}/{obj['Key']}")
        
        print("\n✓ S3 cleanup completed")
        
        # Note: DynamoDB table cleanup requires manual deletion
        print(f"\n⚠️  Manually delete DynamoDB table '{FEATURE_CATALOG_TABLE}' if no longer needed")
        
    except Exception as e:
        print(f"Cleanup error: {e}")
else:
    print("\n📝 Remember to clean up resources manually when done testing")
    print(f"   - S3 objects in s3://{FEATURE_BUCKET}/{FEATURE_PREFIX}")
    print(f"   - DynamoDB table '{FEATURE_CATALOG_TABLE}'")

print("\n🎉 Mini Project 3 Complete!")
print("You've successfully built a feature engineering system using AWS Free Tier services.")